In [1]:
import numpy as np, pandas as pd
from datetime import datetime
from typing import Any, Dict
from pydantic import BaseModel, Field

class Event(BaseModel):
    event_id: int
    timestamp: datetime                 # when it happened
    entity: str                         # who/what it is about (here: a host)
    source_model: str                   # which detector raised it
    threat_type: str                    # BruteForce, PortScan, InsiderMisuse, Ransomware, ...
    raw_score: float = Field(ge=0, le=1)   # detector confidence, normalised to 0-1
    technique: str                      # MITRE ATT&CK ID (verify each on attack.mitre.org)
    label: int = 0                      # ground truth: used ONLY to evaluate, never to score
    context: Dict[str, Any] = {}        # filled by the context engine

In [2]:
rng = np.random.default_rng(11)
T0 = pd.Timestamp("2026-01-01")        # simulation start; day d = T0 + d days
N_HOSTS = 150

net = pd.read_csv("../data/processed/network_scores.csv")
ins = pd.read_csv("../data/processed/insider_scores.csv")
ran = pd.read_csv("../data/processed/ransomware_scores.csv")

ins["pct"] = ins["score"].rank(pct=True)           # convert anomaly score to 0-1 percentile
net_a = net[net["score"] >= 0.5].reset_index(drop=True)
ins_a = ins[ins["pct"] >= 0.95].copy()
ran_a = ran[ran["score"] >= 0.5].copy()
print("raw alerts -> network:", len(net_a), "| insider:", len(ins_a), "| ransomware:", len(ran_a))

bad = ins[ins["label"] == 1].groupby("user")["day"].min()   # user -> first malicious day
insider_users = list(bad.index)                              # the 8 insiders (used for planting)

raw alerts -> network: 127322 | insider: 226 | ransomware: 1275
